# Tech Challenge Fase 1 — Diagnóstico de Diabetes
## 3. Modelagem e Avaliação
Este notebook treina e avalia dois modelos de classificação:
- **Regressão Logística** (modelo baseline interpretável)
- **Random Forest** (modelo ensemble, captura não-linearidades)

A métrica principal é o **Recall**, pois em diagnóstico médico um falso negativo
(não detectar diabetes em quem tem) é mais prejudicial que um falso positivo.

## 3.1 Importação de bibliotecas

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, precision_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

sns.set_theme(style='whitegrid')
print('Bibliotecas importadas com sucesso!')

Bibliotecas importadas com sucesso!


## 3.2 Carregamento dos dados processados

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val   = pd.read_csv('../data/processed/X_val.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_val   = pd.read_csv('../data/processed/y_val.csv').values.ravel()
y_test  = pd.read_csv('../data/processed/y_test.csv').values.ravel()

# Verificar se há NaN
print(f'NaN em X_train: {X_train.isnull().sum().sum()}')
print(f'NaN em X_val:   {X_val.isnull().sum().sum()}')
print(f'NaN em X_test:  {X_test.isnull().sum().sum()}')
print(f'Treino:    {X_train.shape}')
print(f'Validação: {X_val.shape}')
print(f'Teste:     {X_test.shape}')

Treino:    (537, 8)
Validação: (115, 8)
Teste:     (116, 8)


## 3.3 Justificativa da métrica principal

**Por que Recall?**

Em diagnóstico de diabetes:
- **Falso Negativo** = paciente diabético classificado como saudável → não recebe tratamento → risco grave à saúde
- **Falso Positivo** = paciente saudável classificado como diabético → realiza mais exames desnecessários

O custo de um falso negativo é muito maior. Por isso priorizamos o **Recall** (sensibilidade),
que mede a proporção de diabéticos corretamente identificados pelo modelo.

Acompanhamos também **F1-score** e **Accuracy** para ter uma visão completa do desempenho.

## 3.4 Função auxiliar de avaliação

In [23]:
def avaliar_modelo(nome, modelo, X, y, conjunto='Validação'):
    y_pred = modelo.predict(X)
    y_prob = modelo.predict_proba(X)[:, 1] if hasattr(modelo, 'predict_proba') else None

    acc  = accuracy_score(y, y_pred)
    rec  = recall_score(y, y_pred)
    f1   = f1_score(y, y_pred)
    prec = precision_score(y, y_pred)
    auc  = roc_auc_score(y, y_prob) if y_prob is not None else None

    print(f'\n=== {nome} | {conjunto} ===')
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Recall:    {rec:.4f}  ← métrica principal')
    print(f'  F1-score:  {f1:.4f}')
    print(f'  Precision: {prec:.4f}')
    if auc:
        print(f'  ROC-AUC:   {auc:.4f}')

    return {'nome': nome, 'accuracy': acc, 'recall': rec, 'f1': f1, 'precision': prec, 'auc': auc}

print('Função de avaliação definida!')

Função de avaliação definida!


## 3.5 Modelo 1 — Regressão Logística

In [24]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
print('Regressão Logística treinada!')

resultado_lr_val = avaliar_modelo('Regressão Logística', lr, X_val, y_val, 'Validação')

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 3.6 Modelo 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=50, random_state=42,  max_depth=10, n_jobs=-1)
rf.fit(X_train, y_train)
print('Random Forest treinado!')

resultado_rf_val = avaliar_modelo('Random Forest', rf, X_val, y_val, 'Validação')

## 3.7 Comparação dos modelos na validação

In [ ]:
resultados = pd.DataFrame([resultado_lr_val, resultado_rf_val])
resultados = resultados.set_index('nome')
print('=== COMPARAÇÃO NA VALIDAÇÃO ===')
print(resultados[['accuracy', 'recall', 'f1', 'precision', 'auc']].round(4))

# Gráfico comparativo
metricas = ['accuracy', 'recall', 'f1', 'precision']
x = np.arange(len(metricas))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, resultados.loc['Regressão Logística', metricas], width, label='Regressão Logística', color='steelblue')
bars2 = ax.bar(x + width/2, resultados.loc['Random Forest', metricas], width, label='Random Forest', color='tomato')

ax.set_xticks(x)
ax.set_xticklabels(['Accuracy', 'Recall ★', 'F1-score', 'Precision'], fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Comparação de Modelos — Validação', fontsize=14)
ax.legend(fontsize=11)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../notebooks/comparacao_modelos.png', dpi=150)
plt.show()

## 3.8 Avaliação final no conjunto de teste

Avaliamos **ambos os modelos** no teste para transparência, mas o modelo com maior Recall na validação é o escolhido.

In [ ]:
resultado_lr_test = avaliar_modelo('Regressão Logística', lr, X_test, y_test, 'Teste')
resultado_rf_test = avaliar_modelo('Random Forest',       rf, X_test, y_test, 'Teste')

## 3.9 Matriz de confusão

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, modelo, nome in [
    (axes[0], lr, 'Regressão Logística'),
    (axes[1], rf, 'Random Forest')
]:
    cm = confusion_matrix(y_test, modelo.predict(X_test))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Sem Diabetes', 'Com Diabetes'],
        yticklabels=['Sem Diabetes', 'Com Diabetes']
    )
    ax.set_title(f'Matriz de Confusão\n{nome}', fontsize=13)
    ax.set_ylabel('Real')
    ax.set_xlabel('Predito')

plt.tight_layout()
plt.savefig('../notebooks/matriz_confusao.png', dpi=150)
plt.show()

## 3.10 Curva ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for modelo, nome, cor in [
    (lr, 'Regressão Logística', 'steelblue'),
    (rf, 'Random Forest', 'tomato')
]:
    y_prob = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{nome} (AUC={auc:.3f})', color=cor, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Aleatório (AUC=0.500)')
ax.set_xlabel('Taxa de Falsos Positivos', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiros Positivos (Recall)', fontsize=12)
ax.set_title('Curva ROC — Conjunto de Teste', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../notebooks/curva_roc.png', dpi=150)
plt.show()

## 3.11 Relatório de classificação completo

In [ ]:
print('=== REGRESSÃO LOGÍSTICA ===')
print(classification_report(y_test, lr.predict(X_test), target_names=['Sem Diabetes', 'Com Diabetes']))

print('=== RANDOM FOREST ===')
print(classification_report(y_test, rf.predict(X_test), target_names=['Sem Diabetes', 'Com Diabetes']))

## 3.12 Salvar modelos treinados

In [ ]:
import os
os.makedirs('../data/models', exist_ok=True)

with open('../data/models/logistic_regression.pkl', 'wb') as f:
    pickle.dump(lr, f)

with open('../data/models/random_forest.pkl', 'wb') as f:
    pickle.dump(rf, f)

print('Modelos salvos em data/models/')
print(os.listdir('../data/models/'))

## 3.13 Conclusões da Modelagem

**Resumo dos resultados:**

- Ambos os modelos foram treinados e avaliados com Accuracy, Recall, F1-score e AUC-ROC
- A métrica principal escolhida foi o **Recall**, pois em diagnóstico médico minimizar falsos negativos é prioritário
- As matrizes de confusão e a curva ROC permitem visualizar o trade-off entre sensibilidade e especificidade

**Discussão crítica:**

O modelo pode ser usado como ferramenta de **triagem inicial** em ambientes clínicos, ajudando a priorizar pacientes para avaliação médica mais detalhada. Porém, o(a) médico(a) deve sempre ter a palavra final no diagnóstico. O modelo não substitui exames clínicos nem o julgamento profissional.

**Próximo passo:** interpretabilidade dos modelos com Feature Importance e SHAP